# 🔑 PAUTA — ML_U3_Lab02 Regularización y PyTorch

> **Solo para uso del profesor/ayudante. No distribuir a estudiantes.**
> Versión: 2025-1 | Generada: 2026-05-09


# ML_U3_Lab02 — Regularización, Optimización y PyTorch

**Versión:** 2025-1 | **Modificado:** 2026-05-09
**Dataset:** make_classification + load_digits | **Duración:** 1 hora
**Modalidad:** Individual o parejas

---

## 📋 Estructura del laboratorio

| Parte | Tema | Tiempo | Audiencia |
|-------|------|--------|-----------|
| Setup | Imports, datos, funciones auxiliares | 5 min | Todos |
| PARTE 1 | Sin computador: diagrama de Dropout y Adam | 10 min | Todos |
| PARTE 2 | Regularización sistemática con sklearn | 20 min | Todos |
| PARTE 3 | Training loop PyTorch con early stopping | 20 min | Todos |
| ANÁLISIS | Interpretación y reflexión | 5 min | Todos (diferenciado) |

---

## 🎯 Instrucciones por audiencia

| | Pregrado | Doctorado |
|--|----------|-----------|
| Obligatorio | Partes 1, 2, 3 + preguntas azules | Todo lo anterior + TODOs [PhD] + preguntas amarillas |
| Opcional | Bonus azul | Bonus amarillo |
| Entrega | .ipynb ejecutado | .ipynb ejecutado |


## ⚙️ Setup (NO MODIFICAR)

In [ ]:
# ── SETUP — NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_classification, load_digits
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# dataset: make_classification + load_digits  |  generado sintéticamente / sklearn built-in
X_cls, y_cls = make_classification(
    n_samples=600, n_features=20, n_informative=10, n_redundant=5,
    n_classes=3, n_clusters_per_class=1, random_state=RANDOM_STATE
)
digits = load_digits()
X_dig, y_dig = digits.data, digits.target

# PyTorch (opcional, degradación elegante si no está)
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
    print(f"✅ Setup completo | PyTorch {torch.__version__}")
except ImportError:
    TORCH_OK = False
    print("✅ Setup completo | PyTorch no disponible — Parte 3 usa sklearn")

import sklearn
print(f"   numpy {np.__version__} | sklearn {sklearn.__version__}")
print(f"   Clasificación: {X_cls.shape} | Dígitos: {X_dig.shape}")

---
## PARTE 1 — Sin Computador: Razonamiento Sobre Regularización (10 min) 🖊️

Responde en papel antes de ejecutar cualquier celda.


In [ ]:
# ━━━ PARTE 1: VERIFICACIÓN CON CÓDIGO ━━━
# Simulamos las curvas de aprendizaje del ejemplo para visualizar el diagnóstico
epocas    = [20,  50,  100, 150, 200]
train_acc = [0.72, 0.88, 0.97, 0.99, 1.00]
val_acc   = [0.70, 0.83, 0.84, 0.82, 0.80]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epocas, train_acc, 'o-', color='steelblue', label='Train Acc', lw=2)
ax.plot(epocas, val_acc,   'o-', color='tomato',    label='Val Acc',   lw=2)
ax.axvline(x=100, color='gray', linestyle='--', alpha=0.6, label='Inicio sobreajuste (~época 100)')
ax.axvline(x=120, color='green', linestyle=':', alpha=0.6, label='Early stopping (aprox.)')
ax.fill_between(epocas, train_acc, val_acc, alpha=0.08, color='red', label='Brecha train-val')
ax.set_xlabel('Época'); ax.set_ylabel('Accuracy')
ax.set_title('Diagnóstico: Sobreajuste a partir de la época 100')
ax.legend(fontsize=9); ax.set_ylim(0.6, 1.05)
plt.tight_layout(); plt.show()

print("💡 Early stopping patience=20 desde la época 83 (mejor val_acc) → detención ~103")

---
## PARTE 2 — Regularización Sistemática con sklearn (20 min)

Exploración empírica del efecto de las técnicas de regularización sobre el dataset
de clasificación multiclase.


In [ ]:
# ━━━ PAUTA PARTE 2: SPLIT ━━━
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cls, y_cls, test_size=0.25, stratify=y_cls, random_state=RANDOM_STATE
)
print(f"Train: {X_tr.shape} | Test: {X_te.shape}")
print(f"Clases en train: {np.bincount(y_tr)}")

In [ ]:
# ━━━ PAUTA PARTE 2: REGULARIZACIÓN ━━━
configs = {
    'A: Sin regularización':   dict(alpha=0.0,   early_stopping=False),
    'B: L2 moderado (0.001)':  dict(alpha=0.001, early_stopping=False),
    'C: L2 fuerte (0.1)':      dict(alpha=0.1,   early_stopping=False),
    'D: L2 + early_stopping':  dict(alpha=0.001, early_stopping=True,
                                    validation_fraction=0.1),
}
print(f"{'Config':<30} {'CV Acc':>8} {'Std':>7}")
print("-" * 50)
cv_results = {}
for name, kwargs in configs.items():
    pipe = Pipeline([
        ('sc', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(100,50), activation='relu',
                              solver='adam', max_iter=500, random_state=RANDOM_STATE, **kwargs))
    ])
    scores = cross_val_score(pipe, X_cls, y_cls, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:<30} {scores.mean():>8.4f} {scores.std():>7.4f}")

In [ ]:
# ━━━ PAUTA PARTE 2: OPTIMIZADORES ━━━
solvers_config = {
    'SGD (sin momentum)':  dict(solver='sgd', momentum=0.0, learning_rate_init=0.01),
    'SGD + Momentum(0.9)': dict(solver='sgd', momentum=0.9, learning_rate_init=0.01),
    'Adam':                dict(solver='adam'),
}
print(f"{'Optimizador':<25} {'CV Acc':>8} {'Std':>7}")
print("-" * 43)
for name, kwargs in solvers_config.items():
    pipe = Pipeline([
        ('sc', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(100,50), alpha=0.001,
                              max_iter=500, random_state=RANDOM_STATE, **kwargs))
    ])
    scores = cross_val_score(pipe, X_cls, y_cls, cv=5, scoring='accuracy')
    print(f"{name:<25} {scores.mean():>8.4f} {scores.std():>7.4f}")

In [ ]:
# 🔍 Tests de sanidad — Parte 2 (NO MODIFICAR)
try:
    assert X_tr is not None, "TODO 1: split no implementado"
    assert X_tr.shape[0] + X_te.shape[0] == 600, "Split incorrecto"
    print(f"✅ PASS — Split: {X_tr.shape[0]} train / {X_te.shape[0]} test")
except AssertionError as e:
    print(f"❌ FAIL — {e}")

try:
    assert len(cv_results) >= 4, "TODO 2: no hay 4 configuraciones en cv_results"
    for name, scores in cv_results.items():
        assert 0.0 <= scores.mean() <= 1.0, f"{name}: accuracy fuera de rango"
    print("✅ PASS — Comparación de regularización completada")
except (AssertionError, AttributeError, NameError) as e:
    print(f"❌ FAIL — {e}")

---
## PARTE 3 — Training Loop con PyTorch + Early Stopping (20 min)

Implementa un training loop completo en PyTorch para clasificar los dígitos,
incluyendo validación por época y early stopping manual.


In [ ]:
# ━━━ PARTE 3: SETUP PYTORCH (si disponible) ━━━
if not TORCH_OK:
    print("⚠️  PyTorch no disponible — usando sklearn equivalente para los TODOs")
    print("   Instala con: pip install torch")

# Preparar datos de dígitos para PyTorch
X_d, y_d = load_digits().data.astype(np.float32), load_digits().target
X_d_tr, X_d_te, y_d_tr, y_d_te = train_test_split(
    X_d, y_d, test_size=0.2, random_state=RANDOM_STATE, stratify=y_d
)
X_d_tr, X_d_val, y_d_tr, y_d_val = train_test_split(
    X_d_tr, y_d_tr, test_size=0.15, random_state=RANDOM_STATE
)

sc = StandardScaler()
X_d_tr_sc  = sc.fit_transform(X_d_tr).astype(np.float32)
X_d_val_sc = sc.transform(X_d_val).astype(np.float32)
X_d_te_sc  = sc.transform(X_d_te).astype(np.float32)

print(f"Train: {X_d_tr_sc.shape[0]} | Val: {X_d_val_sc.shape[0]} | Test: {X_d_te_sc.shape[0]}")

In [ ]:
# ━━━ PAUTA PARTE 3: ARQUITECTURA ━━━
if TORCH_OK:
    class MLP_Reg(nn.Module):
        def __init__(self, n_in=64, n_h1=128, n_h2=64, n_out=10, dropout_rate=0.3):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_in, n_h1),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(n_h1, n_h2),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(n_h2, n_out)
            )
        def forward(self, x):
            return self.net(x)

    test_model = MLP_Reg()
    test_out = test_model(torch.randn(4, 64))
    print(f"✅ Arquitectura OK — output shape: {test_out.shape}")
    print(test_model)
else:
    print("PyTorch no disponible")

In [ ]:
# ━━━ PAUTA PARTE 3: TRAINING LOOP ━━━
if TORCH_OK:
    N_EPOCHS = 100
    PATIENCE = 15

    X_tr_t  = torch.from_numpy(X_d_tr_sc)
    y_tr_t  = torch.from_numpy(y_d_tr).long()
    X_val_t = torch.from_numpy(X_d_val_sc)
    y_val_t = torch.from_numpy(y_d_val).long()
    X_te_t  = torch.from_numpy(X_d_te_sc)

    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)

    model     = MLP_Reg(dropout_rate=0.3)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    # [PhD] scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    train_losses, val_accs = [], []
    best_val_acc = -1.0
    best_weights = None
    patience_counter = 0
    best_epoch = -1

    for epoch in range(N_EPOCHS):
        # Train
        model.train()
        ep_loss = 0.0
        for bX, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bX), by)
            loss.backward()
            optimizer.step()
            ep_loss += loss.item()
        train_losses.append(ep_loss / len(loader))

        # Validate
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss   = criterion(val_logits, y_val_t).item()
            val_acc    = (val_logits.argmax(1) == y_val_t).float().mean().item()
        val_accs.append(val_acc)

        # [PhD] scheduler step
        scheduler.step(val_loss)

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            best_epoch = epoch
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping en época {epoch+1} (mejor: época {best_epoch+1})")
                model.load_state_dict(best_weights)
                break

    if best_weights and patience_counter < PATIENCE:
        model.load_state_dict(best_weights)
    print(f"Entrenamiento completo: {len(train_losses)} épocas | Mejor val_acc: {best_val_acc:.4f}")
else:
    pipe_fallback = Pipeline([
        ('sc', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(128,64), solver='adam', alpha=1e-4,
                              max_iter=100, early_stopping=True, validation_fraction=0.15,
                              random_state=RANDOM_STATE))
    ])
    pipe_fallback.fit(X_d_tr, y_d_tr)
    train_losses = [0.5]; val_accs = [accuracy_score(y_d_te, pipe_fallback.predict(X_d_te))]
    model = None; best_epoch = 0
    print(f"Fallback sklearn — accuracy test: {val_accs[-1]:.4f}")

In [ ]:
# ━━━ PARTE 3: VISUALIZAR TRAINING ━━━
if TORCH_OK and model is not None and len(train_losses) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(train_losses, color='steelblue', lw=2)
    axes[0].set_title('Pérdida de Entrenamiento'); axes[0].set_xlabel('Época')
    axes[1].plot(val_accs, color='tomato', lw=2)
    if best_epoch >= 0:
        axes[1].axvline(best_epoch, color='green', linestyle='--', alpha=0.7,
                        label=f'Best checkpoint (ép. {best_epoch})')
        axes[1].legend()
    axes[1].set_title('Accuracy Validación'); axes[1].set_xlabel('Época')
    plt.suptitle('Training Loop — MLP con Early Stopping', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
    # Evaluación en test
    model.eval()
    with torch.no_grad():
        preds = model(X_te_t).argmax(dim=1).numpy()
    print(f"✅ Accuracy en test: {accuracy_score(y_d_te, preds):.4f}")

In [ ]:
# 🔍 Tests de sanidad — Parte 3 (NO MODIFICAR)
if TORCH_OK:
    try:
        assert model is not None, "TODO 5: modelo no creado"
        assert hasattr(model, 'net'), "El modelo debe tener atributo 'net'"
        test_out = model(torch.randn(4, 64))
        assert test_out.shape == (4, 10), f"Output shape incorrecto: {test_out.shape}"
        print(f"✅ PASS — Modelo PyTorch creado y funciona correctamente")
    except (AssertionError, TypeError) as e:
        print(f"❌ FAIL — {e}")

    try:
        assert len(train_losses) > 0, "TODO 5: training loop no ejecutado"
        assert len(val_accs) == len(train_losses), "train y val deben tener misma longitud"
        assert all(0 <= a <= 1 for a in val_accs), "val_accs fuera de rango"
        print(f"✅ PASS — Training loop: {len(train_losses)} épocas | Mejor acc: {max(val_accs):.4f}")
    except (AssertionError, NameError) as e:
        print(f"❌ FAIL — {e}")
else:
    print("ℹ️  PyTorch no disponible — tests de sanidad omitidos para Parte 3")

---
## BONUS (Opcional)

---
## ✅ Checklist de Entrega

### Pregrado
- [ ] Parte 1: preguntas 1–3 respondidas a mano
- [ ] TODO 1: split implementado
- [ ] TODO 2: comparación de regularización completada (4 configs)
- [ ] TODO 3: comparación de optimizadores completada
- [ ] TODO 4: arquitectura PyTorch definida
- [ ] TODO 5: training loop con early stopping implementado
- [ ] Preguntas de análisis 1–3 respondidas

### Doctorado (adicional)
- [ ] TODO 5 [PhD]: scheduler ReduceLROnPlateau implementado
- [ ] Parte 1: preguntas 4–6 respondidas
- [ ] Preguntas de análisis 4–6 respondidas
